# FraudShield AI — Notebook 04: Model Training

This notebook trains three models:
1. **Logistic Regression** — baseline linear model
2. **Random Forest** — ensemble, handles non-linearity
3. **XGBoost** — gradient boosting, typically best on tabular fraud data

**Class Imbalance Strategy:**
- `class_weight='balanced'` for Logistic Regression and Random Forest
- XGBoost `scale_pos_weight` parameter
- SMOTE applied only to training data (never test data)

**Prerequisites:** Run notebooks 01, 02, and 03 first.

In [ ]:
import os, warnings, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    confusion_matrix, f1_score, precision_score, recall_score
)
from sklearn.model_selection import cross_val_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')

PROC_PATH  = '../data/processed/'
MODEL_PATH = '../models/'
os.makedirs(MODEL_PATH, exist_ok=True)

# ── Load preprocessed data ────────────────────────────────────────
print('Loading preprocessed data...')
X_train_scaled = np.load(f'{PROC_PATH}X_train.npy')
y_train        = np.load(f'{PROC_PATH}y_train.npy')
X_test_scaled  = np.load(f'{PROC_PATH}X_test.npy')
y_test         = np.load(f'{PROC_PATH}y_test.npy')
X_train_raw    = np.load(f'{PROC_PATH}X_train_raw.npy')
X_test_raw     = np.load(f'{PROC_PATH}X_test_raw.npy')

print(f'X_train: {X_train_scaled.shape}, Fraud rate: {y_train.mean()*100:.2f}%')
print(f'X_test:  {X_test_scaled.shape},  Fraud rate: {y_test.mean()*100:.2f}%')

In [ ]:
# ── SMOTE — Apply ONLY to training data ───────────────────────────
# SMOTE creates synthetic minority (fraud) samples to balance the dataset.
# NEVER apply SMOTE to test data — that would invalidate evaluation.
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
imbalance_ratio = neg / pos
print(f'Training imbalance: {neg:,} legitimate : {pos:,} fraud ({imbalance_ratio:.1f}:1)')

if imbalance_ratio > 10:
    print('Applying SMOTE to balance training data...')
    # Cap sampling_strategy to avoid too many synthetics
    target_ratio = min(0.5, 1.0 / imbalance_ratio * 10)  # max 10x oversample
    smote = SMOTE(sampling_strategy=target_ratio, random_state=42, k_neighbors=5)
    X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)
    X_train_sm_raw, _ = smote.fit_resample(X_train_raw, y_train)
    print(f'After SMOTE: {(y_train_sm==0).sum():,} legitimate : {(y_train_sm==1).sum():,} fraud')
else:
    X_train_sm, y_train_sm = X_train_scaled, y_train
    X_train_sm_raw = X_train_raw
    print('Dataset reasonably balanced. SMOTE not applied.')

In [ ]:
# ── Helper: Quick Evaluation ──────────────────────────────────────
def quick_eval(name, model, X_test, y_test, threshold=0.5):
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= threshold).astype(int)
    return {
        'name':      name,
        'precision': round(precision_score(y_test, preds, zero_division=0), 4),
        'recall':    round(recall_score(y_test, preds, zero_division=0), 4),
        'f1':        round(f1_score(y_test, preds, zero_division=0), 4),
        'roc_auc':   round(roc_auc_score(y_test, probs), 4),
        'pr_auc':    round(average_precision_score(y_test, probs), 4),
    }

In [ ]:
# ── Model 1: Logistic Regression ─────────────────────────────────
print('Training Logistic Regression...')
t0 = time.time()
lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    C=1.0,
    solver='lbfgs',
    random_state=42,
    n_jobs=-1,
)
lr.fit(X_train_sm, y_train_sm)
lr_metrics = quick_eval('Logistic Regression', lr, X_test_scaled, y_test)
print(f'Done in {time.time()-t0:.1f}s | F1={lr_metrics["f1"]} | PR-AUC={lr_metrics["pr_auc"]}')
joblib.dump(lr, f'{MODEL_PATH}logistic_regression.joblib')
print(f'Saved to {MODEL_PATH}logistic_regression.joblib')

In [ ]:
# ── Model 2: Random Forest ────────────────────────────────────────
print('Training Random Forest...')
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train_sm, y_train_sm)
rf_metrics = quick_eval('Random Forest', rf, X_test_scaled, y_test)
print(f'Done in {time.time()-t0:.1f}s | F1={rf_metrics["f1"]} | PR-AUC={rf_metrics["pr_auc"]}')
joblib.dump(rf, f'{MODEL_PATH}random_forest.joblib')
print(f'Saved to {MODEL_PATH}random_forest.joblib')

In [ ]:
# ── Model 3: XGBoost ──────────────────────────────────────────────
# XGBoost uses raw (unscaled) features and handles imbalance via scale_pos_weight
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f'Training XGBoost (scale_pos_weight={scale_pos_weight:.1f})...')
t0 = time.time()
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
xgb.fit(
    X_train_raw, y_train,
    eval_set=[(X_test_raw, y_test)],
    verbose=False,
)
xgb_metrics = quick_eval('XGBoost', xgb, X_test_raw, y_test)
print(f'Done in {time.time()-t0:.1f}s | F1={xgb_metrics["f1"]} | PR-AUC={xgb_metrics["pr_auc"]}')
joblib.dump(xgb, f'{MODEL_PATH}xgboost.joblib')
print(f'Saved to {MODEL_PATH}xgboost.joblib')

In [ ]:
# ── Model Comparison ──────────────────────────────────────────────
results_df = pd.DataFrame([lr_metrics, rf_metrics, xgb_metrics])
print('\nMODEL COMPARISON:')
print(results_df.to_string(index=False))

metrics_to_plot = ['precision','recall','f1','roc_auc','pr_auc']
x = np.arange(len(metrics_to_plot))
width = 0.25
fig, ax = plt.subplots(figsize=(13, 6))
models = [lr_metrics, rf_metrics, xgb_metrics]
colors = ['#3498db','#2ecc71','#e74c3c']
for i, (m, c) in enumerate(zip(models, colors)):
    bars = ax.bar(x + i*width, [m[k] for k in metrics_to_plot], width, label=m['name'], color=c, alpha=0.9)
ax.set_xticks(x + width)
ax.set_xticklabels(['Precision','Recall','F1','ROC-AUC','PR-AUC'])
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontweight='bold', fontsize=14)
ax.legend()
ax.axhline(0.9, linestyle='--', color='gray', alpha=0.5, label='0.9 mark')
plt.tight_layout()
plt.savefig(f'{PROC_PATH}model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved comparison chart to {PROC_PATH}model_comparison.png')

In [ ]:
# ── Training Summary ──────────────────────────────────────────────
best = results_df.sort_values('pr_auc', ascending=False).iloc[0]
print(f'\nBEST MODEL (by PR-AUC): {best["name"]}')
print(f'  Precision : {best["precision"]}')
print(f'  Recall    : {best["recall"]}')
print(f'  F1        : {best["f1"]}')
print(f'  ROC-AUC   : {best["roc_auc"]}')
print(f'  PR-AUC    : {best["pr_auc"]}')
print()
print('Proceed to 05_Model_Evaluation.ipynb for detailed evaluation and model selection.')